# tool-call-agent: fine-tuning a small LLM to emit tool calls

Built during an internship at a telecom company. The question: can a small open model, fine-tuned on a free Colab GPU, turn a plain request like *"Top up line 9 with 50 bucks"* into JSON a Java backend can execute?

```json
{"tool": "chargeLine", "parameters": {"lineId": "9", "amount": 50.0}}
```

**What's in this notebook**

1. Install dependencies and check the GPU
2. Load Gemma 4 E2B (4-bit) and attach LoRA adapters
3. Generate a small synthetic dataset
4. Tokenize and fine-tune
5. Run inference and parse the JSON
6. Evaluate on 30 hand-written test prompts

**Before you run it**

- Use a **T4 GPU** runtime (Runtime > Change runtime type).
- Run the install cell, then **restart the session**, then run everything from the top of section 2.
- All data is synthetic and generated below. Nothing here comes from real customers or company systems.

**Known limits:** 50 training examples, one phrasing, one tool (`chargeLine`). The evaluation in section 6 has not been run yet. This is a prototype, not a finished agent.

## 1. Setup

### Install dependencies

Unsloth patches the model for faster, lower-memory fine-tuning. The `trl` pin is the version that worked for this run. An earlier version of this cell also pinned `xformers<0.0.30`, but that failed to build on current Colab, and Unsloth runs without xformers, so the pin was dropped.

After this cell finishes, restart the session (Runtime > Restart session, or Ctrl+M .) so Python picks up the new packages.

> **Note:** on my original run the `unsloth_zoo` import failed and I patched a few missing functions into the package by hand. I removed that hack from this version. If you hit `ImportError`s from `unsloth_zoo` on a newer or older release, reinstalling both `unsloth` and `unsloth-zoo` from GitHub with `--no-cache-dir` was what fixed it for me.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.15.0" peft accelerate bitsandbytes

### Check the GPU

Colab's free tier gives a Tesla T4 (about 15 GB). The T4 has no bfloat16 support, which is why the training arguments below set fp16. On the 2026-11-27 run, Unsloth reported that fp16 does not work for Gemma 4 and ran it in float32 instead.

In [ ]:
!nvidia-smi

## 2. Load the model and add LoRA adapters

**Base model:** `unsloth/gemma-4-E2B-it-unsloth-bnb-4bit`, Gemma 4 E2B instruction tuned, quantized to 4 bits so it fits comfortably on a T4.

**LoRA** trains small adapter matrices instead of all the model weights. Here: rank 16, alpha 16, no dropout, applied to the attention (`q/k/v/o_proj`) and MLP (`gate/up/down_proj`) layers. `use_gradient_checkpointing="unsloth"` trades a little speed for much lower memory use.

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it-unsloth-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("Gemma 4 E2B loaded with LoRA adapters")

## 3. Synthetic training data

The agent is meant to call Java methods. Three tools are defined in the project:

| Tool | Parameters |
|---|---|
| `chargeLine` | `lineId`, `amount` |
| `checkLineStatus` | `lineId` |
| `emergencyStop` | `lineId` |

Only `chargeLine` appears in the training data below, so the model has not been trained on the other two.

Each example is 50 generated requests of the form *"Charge {amt} to line {id}"*. The target output is a one-line reasoning step, then the JSON action:

```
<|think|>
Action: chargeLine. ID: 12. Amt: 50.
<|channel|>{"tool": "chargeLine", "parameters": {"lineId": "12", "amount": 50.0}}
```

The prompt tokens (`<|turn|>`, `<|think|>`, `<|channel|>`) are written by hand here, so they may not match Gemma's official chat template exactly. Training and inference use the same hand-written format, which is what matters for this experiment.

In [ ]:
import json
import random
from collections import Counter
from datasets import Dataset

random.seed(3407)

SYSTEM_PROMPT = (
    "Agent. Tools: chargeLine(lineId, amount), checkLineStatus(lineId), "
    "emergencyStop(lineId), none(reason)."
)

AMOUNTS = [5, 10, 15, 20, 25, 30, 40, 50, 60, 75, 100, 150, 200]
CURRENCIES = ["bucks", "euros", "units"]

CHARGE_TEMPLATES = [
    "Charge {amt} to line {id}",
    "Top up line {id} with {amt}",
    "Add {amt} credit to line {id}",
    "Put {amt} on line {id}",
    "Line {id} needs {amt} more",
    "Charge {amt} {cur} to line {id}",
    "Top up line {id} with {amt} {cur}",
    "Put {amt} {cur} on line {id}",
    "Line {id} needs {amt} {cur} more",
]
STATUS_TEMPLATES = [
    "Is line {id} active?",
    "Check status of line {id}",
    "What's going on with line {id}",
    "Status for line {id} please",
    "Is {id} working?",
]
STOP_TEMPLATES = [
    "Stop line {id} now",
    "Emergency stop on line {id}",
    "Kill line {id} immediately",
    "Shut down line {id}, it is compromised",
    "Cut line {id} right away",
]
MISSING_ID_TEMPLATES = [
    "Top up {amt}",
    "Charge {amt} please",
    "Add {amt} {cur} of credit",
    "Put {amt} on my line",
]
MISSING_AMOUNT_TEMPLATES = [
    "Charge line {id}",
    "Top up line {id}",
    "Add credit to line {id}",
    "Put some money on line {id}",
]
UNSUPPORTED_PROMPTS = [
    "What's the weather in Amman?",
    "Write me a poem",
    "Tell me a joke",
    "Who won the football match yesterday?",
    "Translate good morning into Spanish",
    "What time is it in Tokyo?",
    "Recommend a good movie",
    "How do I bake bread?",
    "Explain how a black hole works",
    "Help me write a cover letter",
    "What's 17 times 23?",
    "Order me a pizza",
]


def fill(template):
    return template.format(id=random.randint(1, 99),
                           amt=random.choice(AMOUNTS),
                           cur=random.choice(CURRENCIES))


def charge_example():
    line_id = str(random.randint(1, 99))
    amount = random.choice(AMOUNTS)
    template = random.choice(CHARGE_TEMPLATES)
    return {
        "content": template.format(id=line_id, amt=amount, cur=random.choice(CURRENCIES)),
        "thought": f"Action: chargeLine. ID: {line_id}. Amt: {amount}.",
        "action": {"tool": "chargeLine",
                   "parameters": {"lineId": line_id, "amount": float(amount)}},
    }


def line_example(tool, templates):
    line_id = str(random.randint(1, 99))
    return {
        "content": random.choice(templates).format(id=line_id),
        "thought": f"Action: {tool}. ID: {line_id}.",
        "action": {"tool": tool, "parameters": {"lineId": line_id}},
    }


def none_example(content, reason):
    return {
        "content": content,
        "thought": f"Action: none. Reason: {reason}.",
        "action": {"tool": "none", "parameters": {"reason": reason}},
    }


dataset_samples = []
dataset_samples += [charge_example() for _ in range(80)]
dataset_samples += [line_example("checkLineStatus", STATUS_TEMPLATES) for _ in range(60)]
dataset_samples += [line_example("emergencyStop", STOP_TEMPLATES) for _ in range(60)]
dataset_samples += [none_example(fill(random.choice(MISSING_ID_TEMPLATES)), "missing_line_id")
                    for _ in range(14)]
dataset_samples += [none_example(fill(random.choice(MISSING_AMOUNT_TEMPLATES)), "missing_amount")
                    for _ in range(14)]
dataset_samples += [none_example(p, "unsupported_request") for p in UNSUPPORTED_PROMPTS]
random.shuffle(dataset_samples)

assert len(dataset_samples) == 240
print(Counter(s["action"]["tool"] for s in dataset_samples))


def format_chat(samples):
    return [{"text": (
        f"<bos><|turn|>system\n{SYSTEM_PROMPT}<|turn|>user\n{s['content']}"
        f"<|turn|>assistant\n<|think|>\n{s['thought']}\n"
        f"<|channel|>{json.dumps(s['action'])}<eos>"
    )} for s in samples]


dataset = Dataset.from_list(format_chat(dataset_samples))
print(dataset[0]["text"])

## 4. Tokenize and train

**Tokenizing:** examples are padded or truncated to 128 tokens. The pad token is set to the end-of-sequence token. Unsloth wraps the tokenizer, so `tokenizer_only` pulls out the underlying Hugging Face tokenizer.

**Training:** the plain Hugging Face `Trainer`, not Unsloth's trainer, because the Unsloth chat-template route failed for me on this setup.

| Setting | Value |
|---|---|
| Batch size | 1, with 8 gradient accumulation steps (effective batch of 8) |
| Steps | 60 (about 10 passes over 50 examples) |
| Learning rate | 2e-4 |
| Optimizer | `adamw_8bit` |
| Precision | `fp16=True` in the arguments, but Unsloth ran Gemma 4 in float32 |

Watch the loss in the logs. On a dataset this small it should fall quickly.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

tokenizer_only = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
tokenizer_only.pad_token = tokenizer_only.eos_token

# padding and truncating to 128 would silently cut off long examples, so check first
lengths = [len(tokenizer_only(t)["input_ids"]) for t in dataset["text"]]
print(f"Longest example: {max(lengths)} tokens")
assert max(lengths) <= 128, "an example is longer than max_length and would be truncated"

def tokenize(element):
    return tokenizer_only(element["text"], truncation=True,
                          max_length=128, padding="max_length")

tokenized_dataset = dataset.map(tokenize, batched=True,
                                remove_columns=dataset.column_names)

In [ ]:
trainer = Trainer(
    model = model,
    train_dataset = tokenized_dataset,
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer_only, mlm=False),
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        max_steps = 90,
        learning_rate = 2e-4,
        fp16 = True,
        optim = "adamw_8bit",
        gradient_checkpointing = True,
        gradient_checkpointing_kwargs = {"use_reentrant": False},
        output_dir = "tool_call_agent_v2",
        report_to = "none",
        logging_steps = 1,
    ),
)

trainer.train()

## 5. Inference

Free-form prompting gave rambling output from this model, not clean JSON. I went through several fixes while debugging:

1. Greedy decoding plus a stop string
2. Priming the answer with `Action:`
3. A raw `USER: ... AGENT_ACTION:` format with a repetition penalty
4. Pre-filling the response up to the opening `{` of the JSON
5. Adding a format hint and a one-shot example to the system prompt

The version below combines the last two. It pre-fills `Action: chargeLine.` and the opening brace, so the model only has to produce the line ID and amount, then stops at the closing `}`.

**Limitation:** because the tool name is pre-filled, the model is not choosing the tool. Removing that pre-fill is the main next step (see the README).

In [ ]:
FastLanguageModel.for_inference(model)

def generate_raw(user_input):
    # pre-fill only up to "Action:" so the model writes the tool name itself
    prompt = (
        f"<bos><|turn|>system\n{SYSTEM_PROMPT}<|turn|>user\n{user_input}"
        f"<|turn|>assistant\n<|think|>\nAction:"
    )
    inputs = tokenizer_only(prompt, return_tensors="pt",
                            add_special_tokens=False).to("cuda")
    outputs = model.generate(
        input_ids = inputs.input_ids,
        attention_mask = inputs.attention_mask,
        max_new_tokens = 80,
        do_sample = False,          # greedy decoding
        eos_token_id = tokenizer_only.eos_token_id,
    )
    new_tokens = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer_only.decode(new_tokens, skip_special_tokens=False)

def extract_json(text):
    # take the text after the first <|channel|> and return the first balanced {...}
    marker = "<|channel|>"
    if marker not in text:
        return None
    text = text.split(marker, 1)[1]
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return None

def tool_call_agent(user_input):
    return extract_json(generate_raw(user_input))

### Try it

Edit the prompts to test your own. The first one is phrased differently from anything in the training data, so it's a quick check of how well the model handles new wording.

In [ ]:
test_prompts = [
    "Top up line 9 with 50 bucks.",
    "Is line 31 up and running?",
    "Shut off line 44, someone stole the phone.",
    "Add 20 euros please.",
]

for p in test_prompts:
    print(f"INPUT:  {p}")
    print(f"OUTPUT: {tool_call_agent(p)}\n")

### Parse the output

The backend needs valid JSON, so the last step is to parse and sanity check the result before passing it on. Anything that fails here should be rejected, not executed.

In [ ]:
import math
import re

TOOL_SCHEMAS = {
    "chargeLine": {"lineId", "amount"},
    "checkLineStatus": {"lineId"},
    "emergencyStop": {"lineId"},
    "none": {"reason"},
}
NONE_REASONS = {"missing_line_id", "missing_amount", "unsupported_request"}

def parse_action(raw):
    if raw is None:
        return None
    try:
        action = json.loads(raw)
    except json.JSONDecodeError:
        return None
    if not isinstance(action, dict) or set(action) != {"tool", "parameters"}:
        return None
    tool, params = action["tool"], action["parameters"]
    if tool not in TOOL_SCHEMAS or not isinstance(params, dict):
        return None
    if set(params) != TOOL_SCHEMAS[tool]:
        return None
    if "lineId" in params:
        line_id = params["lineId"]
        if not isinstance(line_id, str) or not re.fullmatch(r"[0-9]+", line_id):
            return None
    if "amount" in params:
        amount = params["amount"]
        if isinstance(amount, bool) or not isinstance(amount, (int, float)):
            return None
        if not math.isfinite(amount) or amount <= 0:
            return None
    if tool == "none" and params["reason"] not in NONE_REASONS:
        return None
    return action

for p in test_prompts:
    print(p, "->", parse_action(tool_call_agent(p)))

## What's next

- Train on all three tools, plus vague and invalid requests the agent should refuse or ask about.
- Run the evaluation in section 6 and report exact-match accuracy on tool name and arguments.
- Remove the pre-filled `Action: chargeLine.` so the model picks the tool itself.
- Use constrained decoding against a JSON schema.
- Save the LoRA adapter (`model.save_pretrained(...)`) or export a merged model, and call it from a mock Java service.

## 6. Evaluation

`data/test_prompts.jsonl` holds 30 hand-written `chargeLine` requests with the expected JSON for each. The wording is much more varied than the training data ("top up", "add credit", "put N on line X", "bucks", "euros", different word order), and some amounts (15, 75, 200, 250, 500) never appear in training.

For each prompt the cells below run `tool_call_agent`, then `parse_action`, and count:

- how many outputs are valid JSON
- how many match the expected tool, `lineId` and `amount` exactly

Failures are listed in a table at the end. The first cell clones the repo if the test file isn't already next to the notebook, which is the usual case in Colab.

In [ ]:
import os
import subprocess

BRANCH = "v2"
REPO_URL = "https://github.com/mohammadhamayel23/tool-call-agent.git"

TEST_FILE = "data/test_prompts.jsonl"
if not os.path.exists(TEST_FILE):
    clone_dir = f"tool-call-agent-{BRANCH}"
    if not os.path.exists(clone_dir):
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        REPO_URL, clone_dir], check=True)
    TEST_FILE = f"{clone_dir}/data/test_prompts.jsonl"

with open(TEST_FILE, encoding="utf-8") as f:
    test_cases = [json.loads(line) for line in f if line.strip()]
print(f"Loaded {len(test_cases)} test cases from {TEST_FILE}")

In [ ]:
train_prompts = {s["content"].strip() for s in dataset_samples}
overlap = [c["prompt"] for c in test_cases if c["prompt"].strip() in train_prompts]
assert not overlap, f"test prompts found in training data: {overlap}"
print("No test prompt appears verbatim in the training prompts")

In [ ]:
import pandas as pd

rows = []
for case in test_cases:
    raw_text = generate_raw(case["prompt"])
    extracted = extract_json(raw_text)
    try:
        json.loads(extracted)
        valid_json = True
    except (TypeError, json.JSONDecodeError):
        valid_json = False
    action = parse_action(extracted)
    expected = case["expected"]
    tool_correct = action is not None and action["tool"] == expected["tool"]
    exact = action is not None and action == expected
    rows.append({
        "prompt": case["prompt"],
        "expected_tool": expected["tool"],
        "expected": json.dumps(expected),
        "output": extracted if extracted is not None else raw_text,
        "valid_json": valid_json,
        "tool_correct": tool_correct,
        "exact": exact,
    })

results = pd.DataFrame(rows)
n = len(results)
print(f"Valid JSON:    {results['valid_json'].sum()}/{n}")
print(f"Tool accuracy: {results['tool_correct'].sum()}/{n}")
print(f"Exact match:   {results['exact'].sum()}/{n} (tool and all parameters)")

In [ ]:
per_tool = (results.groupby("expected_tool")
            .agg(n=("prompt", "size"),
                 tool_correct=("tool_correct", "sum"),
                 exact=("exact", "sum"))
            .reindex(["chargeLine", "checkLineStatus", "emergencyStop", "none"]))
per_tool

In [ ]:
failures = results[~results["exact"]]
print(f"{len(failures)} failures")
pd.set_option("display.max_colwidth", None)
failures[["prompt", "expected", "output"]]